# A100: M2 SegFormer seeds 1-2 + two tuning variants
Runtime: **A100 GPU**. *Runtime > Run all*, approve Drive, leave the tab open. About 2-2.5 h, roughly 25-30 units.
If the account has fewer units, delete the last run (ADE20K) before Run all.

Order: seeds first (needed for a fair 3-seed comparison), then the "gentle" variant (targets the early overfitting seen
in seed 0), then the ADE20K-pretrained encoder (optional). Run names differ from every other run, so nothing is overwritten.
A failed run does not stop the queue. Results go to `checkpoints/` on the shared Drive.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # expect A100
import os; print("CPU cores:", os.cpu_count())   # 8 workers below assumes >= 9 cores; lower it if fewer

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
CPU cores: 12


In [2]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
!ls "{CK}"

Mounted at /content/drive
m0_early_fusion_unet_s0  m1_siamese_unet_s2
m0_early_fusion_unet_s1  m2_siamese_segformer_ade_ce_s0
m0_early_fusion_unet_s2  m2_siamese_segformer_ce_s0
m1_siamese_unet_ce_s0	 m2_siamese_segformer_ce_s1
m1_siamese_unet_ce_s1	 m2_siamese_segformer_ce_s2
m1_siamese_unet_ce_s2	 m2_siamese_segformer_gentle_ce_s0
m1_siamese_unet_s0	 smoke_m1_s0
m1_siamese_unet_s1	 smoke_m2


In [3]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [4]:
# Data to local disk (10-15 min)
!mkdir -p /content/xbd
!tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
!tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
!cp -r "{D}/xbd/hold" /content/xbd/
!cp -r "{D}/xbd/tier3" /content/xbd/
!cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd

hold  rscc_flood_manifest.csv  test  tier3  train


## 1-2. M2 seeds 1 and 2: same recipe as seed 0 (weighted CE, encoder lr 6e-5, decoder lr 6e-4, 20 epochs)
About 10 min into the first run, check `nvidia-smi --query-gpu=utilization.gpu --format=csv -l 2` in the Terminal:
mostly high = the A100 is fed.

In [5]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=m2_siamese_segformer_ce_s1 seed=1 loss=ce lr=6e-4 encoder_lr_mult=0.1 num_workers=8

run m2_siamese_segformer_ce_s1: train 5488 tiles, val 2184 tiles, device cuda, amp True
config.json: 100% 70.0k/70.0k [00:00<00:00, 56.6MB/s]

pytorch_model.bin: downloading bytes:  93% 50.7M/54.7M [00:01<00:00, 62.5MB/s, 3.14MB/s  ]
pytorch_model.bin: downloading bytes: 100% 51.4M/51.4M [00:02<00:00, 25.3MB/s, 4.83MB/s  ]
pytorch_model.bin: reconstructing file: 100% 54.7M/54.7M [00:02<00:00, 27.0MB/s, 5.20MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 17693.99it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b1
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors: downloading bytes:  90% 49.1M/54.7M [00:02<00:00, 45.3MB/s, 3.42MB/s  ]
model.safetensors: downloading bytes: 100% 51.3M/51.3M [00:02<00:00, 20.6MB/s, 4.75MB/s  ]
model.safe

In [5]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=m2_siamese_segformer_ce_s2 seed=2 loss=ce lr=6e-4 encoder_lr_mult=0.1 num_workers=10


run m2_siamese_segformer_ce_s2: train 5488 tiles, val 2184 tiles, device cuda, amp True
config.json: 100% 70.0k/70.0k [00:00<00:00, 72.0MB/s]

pytorch_model.bin: downloading bytes:  83% 45.3M/54.7M [00:01<00:00, 56.8MB/s, 2.89MB/s  ]
pytorch_model.bin: downloading bytes: 100% 51.4M/51.4M [00:01<00:00, 26.4MB/s, 4.85MB/s  ]
pytorch_model.bin: reconstructing file: 100% 54.7M/54.7M [00:01<00:00, 28.1MB/s, 5.22MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 19984.77it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b1
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors: downloading bytes:  77% 42.1M/54.7M [00:01<00:00, 53.8MB/s, 2.39MB/s  ]
model.safetensors: downloading bytes: 100% 51.3M/51.3M [00:01<00:00, 28.7MB/s, 4.88MB/s  ]
model.safe

## 3. Tuning variant "gentle": 5x lower learning rates (encoder 1.2e-5, decoder 1.2e-4) and 10 epochs
Seed 0 peaked at epoch 2 and then overfit (destroyed F1 collapsed). Slower learning over a shorter schedule targets that.
Exploration only: judged on Florence, reported separately from the 3-seed comparison.

In [6]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=m2_siamese_segformer_gentle_ce_s0 loss=ce lr=1.2e-4 encoder_lr_mult=0.1 epochs=10 warmup_steps=200 num_workers=10

run m2_siamese_segformer_gentle_ce_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
Loading weights: 100% 192/192 [00:00<00:00, 18243.95it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b1
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  epoch 0 step 50/686 loss 1.3601 (running 1.5722)
  epoch 0 step 100/686 loss 1.1623 (running 1.4638)
  epoch 0 step 150/686 loss 0.6768 (running 1.3155)
  epoch 0 step 200/686 loss 0.9411 (running 1.1691)
  epoch 0 step 250/686 loss 0.7273 (running 1.0740)
  epoch 0 step 300/686 loss 1.0207 (running 0.9928)
  epoch 0 step 350/686 loss 0.3193 (running 0.9194)
  epoch 0 step 400/686 loss 0.5800 (running 0.8615)
  epoch 0 step 450/686 loss 0.3418 (running 0.8096)
  epoch 0 step 500/686 loss 0.3153 (running 0.769

## 4. Tuning variant (optional): encoder pretrained for segmentation (ADE20K) instead of ImageNet classification
Expect a load report listing `decode_head...` weights as unused: we only take the encoder. Paste any ERROR.

In [7]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=m2_siamese_segformer_ade_ce_s0 backbone=nvidia/segformer-b1-finetuned-ade-512-512 loss=ce lr=6e-4 encoder_lr_mult=0.1 num_workers=10

run m2_siamese_segformer_ade_ce_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
config.json: 100% 6.88k/6.88k [00:00<00:00, 15.4MB/s]

pytorch_model.bin: downloading bytes:   2% 864k/54.9M [00:01<01:41, 531kB/s]
pytorch_model.bin: downloading bytes:  33% 18.4M/54.9M [00:01<00:02, 14.7MB/s, 1.05MB/s  ]
pytorch_model.bin: downloading bytes:  46% 25.2M/54.9M [00:02<00:01, 18.8MB/s, 2.09MB/s  ]
pytorch_model.bin: downloading bytes:  86% 47.4M/54.9M [00:02<00:00, 37.4MB/s, 3.82MB/s  ]
pytorch_model.bin: downloading bytes: 100% 51.5M/51.5M [00:02<00:00, 18.1MB/s, 4.64MB/s  ]
pytorch_model.bin: reconstructing file: 100% 54.9M/54.9M [00:02<00:00, 19.3MB/s, 4.99MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 24136.99it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/segformer-b1-finetuned-ade-512-512
Key                                           | Status     |  | 
----------------------------------------------+------------+--+-
decode_head.batch_norm.weight            

In [8]:
import json, os, pandas as pd
CK = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/checkpoints"
FAMILIES = {
    "M1 weighted CE":      "m1_siamese_unet_ce",
    "M1 Dice+focal":       "m1_siamese_unet",
    "M0 Dice+focal":       "m0_early_fusion_unet",
    "M2 SegFormer CE":     "m2_siamese_segformer_ce",
    "M2 gentle (variant)": "m2_siamese_segformer_gentle_ce",
    "M2 ADE20K (variant)": "m2_siamese_segformer_ade_ce",
}
rows, curves = [], {}
for name, pre in FAMILIES.items():
    for s in range(3):
        p = f"{CK}/{pre}_s{s}/metrics.jsonl"
        if not os.path.exists(p):
            continue
        df = pd.DataFrame([json.loads(l) for l in open(p)])
        b = df.loc[df["xview2_pixel_score"].idxmax()]
        last = df.iloc[-1]
        rows.append({"model": name, "seed": s, "epochs": len(df), "best_ep": int(b.epoch),
                     "score": b.xview2_pixel_score, "loc": b.loc_f1, "minor": b.dmg_f1_minor,
                     "major": b.dmg_f1_major, "destroyed": b.dmg_f1_destroyed,
                     "last_score": last.xview2_pixel_score, "last_destroyed": last.dmg_f1_destroyed})
        if pre.startswith("m2"):
            curves[f"{pre}_s{s}"] = df.set_index("epoch")["xview2_pixel_score"].round(3)
T = pd.DataFrame(rows)
print(T.round(3).to_string(index=False))
print()
print(T.groupby("model", sort=False)[["score", "loc", "minor", "major", "destroyed"]].agg(["mean", "std", "count"]).round(3))
print()
print("M2 validation score per epoch:")
print(pd.DataFrame(curves).to_string())

              model  seed  epochs  best_ep  score   loc  minor  major  destroyed  last_score  last_destroyed
     M1 weighted CE     0      20       12  0.363 0.782  0.100  0.449      0.124       0.356           0.087
     M1 weighted CE     1      20       12  0.363 0.803  0.086  0.519      0.123       0.349           0.092
     M1 weighted CE     2      20       18  0.356 0.803  0.100  0.532      0.090       0.353           0.087
      M1 Dice+focal     0      20        9  0.316 0.794  0.112  0.425      0.043       0.293           0.027
      M1 Dice+focal     1      20       14  0.326 0.819  0.107  0.524      0.045       0.310           0.034
      M1 Dice+focal     2      20       11  0.313 0.787  0.099  0.598      0.044       0.270           0.012
      M0 Dice+focal     0      20        9  0.286 0.756  0.080  0.507      0.032       0.249           0.006
      M0 Dice+focal     1      20        7  0.274 0.760  0.081  0.544      0.022       0.264           0.011
      M0 Dice+focal

## Stop billing

In [9]:
from google.colab import runtime
runtime.unassign()